# CorePromoter model clean workflow

This notebook is a simplified version of Model_CorePromoter_v0.ipynb. It keeps the core workflow only:

1. setup and utilities
2. training-data assembly
3. model definition
4. train/test split and training
5. save the trained checkpoint
6. train/test evaluation
7. trained filter logos
8. NN-based assembled-sequence scan

BPM scanning and old debug cells are intentionally removed.


In [ ]:
## Setup
import os
import random
import itertools
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

import logomaker
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import train_test_split

from util import getPFM, Motif2Seqs, sci_ticks

SEED = 777

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Utility functions

These helper functions are shared by training and scan cells.


In [ ]:
def stratified_sample(df, col, counts, seed=SEED):
    df = df.copy()
    rng_bins = pd.cut(df[col], bins=len(counts))
    parts = []
    for grp, cnt in zip(sorted(rng_bins.dropna().unique()), counts):
        sub = df[rng_bins == grp]
        if len(sub) == 0:
            continue
        parts.append(sub.sample(cnt, replace=True, random_state=seed))
    return pd.concat(parts)


def dna_one_hot(seq, flatten=False):
    # Keep this mapping identical to dna_one_hot in
    # recursive_corepromoter_design.py. That module scores the designed sequences;
    # this one builds the training tensors. If the two ever disagree, the model is
    # trained in one encoding and used in another.
    mapping = {
        'A': [1, 0, 0, 0],
        'C': [0, 1, 0, 0],
        'G': [0, 0, 1, 0],
        'T': [0, 0, 0, 1],
        # Padding means "no sequence here", so it must contribute nothing. The
        # previous [-1, -1, -1, -1] made conv1's output at that position the NEGATED
        # column sum, and nothing kept that sum positive: training could lower it and
        # turn padding into a score bonus. Only UL() carries dashes, and UL is the one
        # library whose variable region sits under conv1 filter 1, so the model could
        # cut UL's loss by reading padding instead of learning UP -- and those
        # distorted weights then scored designed sequences containing no dashes.
        # Measured before the fix: all eight filters scored an all-dash window
        # positively (+2.8 to +10.7).
        # [0, 0, 0, 0], not [1/4]*4: "absent", not "any base".
        '-': [0, 0, 0, 0],
        'N': [1/4, 1/4, 1/4, 1/4],
    }
    one_hot = np.array([mapping.get(base, [0, 0, 0, 0]) for base in seq.upper()]).T
    return one_hot.flatten() if flatten else one_hot


## Build training dataset

The dataframe name is kept as `df_train` for compatibility, but it represents the full sampled dataset before train/test split.


In [ ]:
TABLE_DIR = "../tables"
LIBRARY_ORDER = ["PL17", "SL16", "SL17", "SL18", "DL", "UL", "ITS"]


def PpurR(seq):
    return 'CACAGCAGCAGTCAGGTACTCCAGTCCAC' + seq[:6] + 'TTTTCGTCAAGATCGGC' + seq[-6:] + 'TCCACGCTTACTAAATCTATGT'


def SL16(seq):
    return 'ACAGCAGCAGTCAGGTACTCCAGTAAATGCTTGACT' + seq + 'TATTATGCACACCCCTAAATCTATGTAG'


def SL17(seq):
    return 'CACAGCAGCAGTCAGGTACTCCAGTCCACTTGACC' + seq + 'TAACCTTCCACGCTTACTAAATCTATGT'


def SL18(seq):
    return 'TCTTCACACAGCAGCTCAGGTACTCAGGCTTTACA' + seq + 'GATAATGTGTGGAATTAAATCTATGTA'


def DL(seq):
    return 'AGCAGCAGTCAGGTACTCCAGTAAATGCTTGCCCGCCGCGTGATTCGTGTTATAAC' + seq + 'CAAAATCTATGTAGCT'


def UL(seq):
    # return 'CAGAAAAAG' + seq + 'GGCTTGCGGCTTTTGCCGCTTTTTTTTACCCTGCACACCCCT----------'
    return 'CAGAAAAAG'+seq+'GGCTTGCGGCTTTTGCCGCTTTTTTTTACCCTGCACACCCCTAAATCTATGT' # 80 bp


def ITS(seq):
    return 'AGCAGCGTTAAATTCACGCCCTTCTCTTGAGACATTTCTTTTGCACTGGTAAACTAAATC' + seq + 'GTCCCAGGCT'


def build_training_dataframe():
    df_PL17 = stratified_sample(pd.read_pickle(f"{TABLE_DIR}/PL.pkl"), 'LogGFP', [2000, 2000, 2000, 1000, 1000, 1000, 500, 500])[['rep1', 'rep2', 'LogGFP']]
    df_PL17['Sequence'] = df_PL17.index.map(PpurR)
    df_PL17['Library'] = "PL17"

    df_SL16 = stratified_sample(pd.read_pickle(f"{TABLE_DIR}/SL16.pkl"), 'LogGFP', [300] * 10)[['rep1', 'rep2', 'LogGFP']]
    df_SL16['Sequence'] = df_SL16.index.map(SL16)
    df_SL16['Library'] = "SL16"

    df_SL17 = stratified_sample(pd.read_pickle(f"{TABLE_DIR}/SL17.pkl"), 'LogGFP', [400] * 10)[['rep1', 'rep2', 'LogGFP']]
    df_SL17['Sequence'] = df_SL17.index.map(SL17)
    df_SL17['Library'] = "SL17"

    df_SL18 = stratified_sample(pd.read_pickle(f"{TABLE_DIR}/SL18.pkl"), 'LogGFP', [300] * 10)[['rep1', 'rep2', 'LogGFP']]
    df_SL18['Sequence'] = df_SL18.index.map(SL18)
    df_SL18['Library'] = "SL18"

    df_DL = stratified_sample(pd.read_pickle(f"{TABLE_DIR}/DL.pkl"), 'LogGFP', [200] * 10)[['rep1', 'rep2', 'LogGFP']]
    df_DL['Sequence'] = df_DL.index.map(DL)
    df_DL['Library'] = "DL"

    df_UL = pd.read_pickle(f"{TABLE_DIR}/UL.pkl")
    df_UL = df_UL[(df_UL['LogGFP'] > 1) & (df_UL['LogGFP'] < 4)]
    df_UL = stratified_sample(df_UL, 'LogGFP', [500, 1000, 1500, 1500, 1000, 500])[['rep1', 'rep2', 'LogGFP']]
    df_UL['Sequence'] = df_UL.index.map(UL)
    df_UL['Library'] = "UL"

    df_ITS = pd.read_pickle(f"{TABLE_DIR}/ITS.pkl")
    df_ITS = stratified_sample(df_ITS, 'LogGFP', [500, 1000, 1500, 1500, 1000, 500])[['rep1', 'rep2', 'LogGFP']]
    df_ITS['Sequence'] = df_ITS.index.map(ITS)
    df_ITS['Library'] = "ITS"

    return pd.concat([df_PL17, df_SL16, df_SL17, df_SL18, df_DL, df_UL, df_ITS]).reset_index(drop=True)


df_train = build_training_dataframe()
print(df_train.shape)
display(df_train.groupby('Library').size().reindex(LIBRARY_ORDER))
df_train.head()


## Model definition

The model keeps the original structure: 8 first-layer filters and 3 architecture channels for spacer flexibility.


In [ ]:
class DNAFunctionPredictor(nn.Module):
    def __init__(self, seq_length, num_conds):
        super().__init__()
        self.seq_length = seq_length
        self.conv1 = nn.Conv1d(in_channels=4, out_channels=8, kernel_size=8, stride=1)
        self._initialize_conv1()

        self.conv2 = nn.Conv1d(in_channels=8, out_channels=3, kernel_size=65, stride=1)
        self._initialize_conv2()

        self.conds_bias = nn.Parameter(torch.zeros(num_conds))
        self.param_max = nn.Parameter(torch.tensor(np.log(1000.0)), requires_grad=True)
        self.param_min = nn.Parameter(torch.tensor(np.log(0.5)), requires_grad=True)
        self.param_e0 = nn.Parameter(torch.tensor(0.0), requires_grad=True)

    def _initialize_conv1(self):
        motifs = [
            "AAAATTTG",
            "GAAAATAG",
            "TTGACATT",
            "NGGCCTAA",
            "ATGGGGTA",
            "TAATTTTT",
            "AAAGCAAA",
            "AAAAAGNN",
        ]
        custom_weights = np.stack([
            getPFM(Motif2Seqs(motif), ratios=True).values.T for motif in motifs
        ], axis=0)
        self.conv1.weight = nn.Parameter(torch.tensor(custom_weights, dtype=torch.float32), requires_grad=True)
        self.conv1.bias.data.zero_()
        self.conv1.bias.requires_grad = False

    def _initialize_conv2(self):
        filter1_weights = np.zeros((8, 65))
        filter2_weights = np.zeros((8, 65))
        filter3_weights = np.zeros((8, 65))

        filter1_weights[0, 3], filter1_weights[1, 11], filter1_weights[2, 22], filter1_weights[3, 30], filter1_weights[4, 38], filter1_weights[5, 46], filter1_weights[6, 54], filter1_weights[7, 62] = 1, 1, 1, 1, 1, 1, 1, 1
        filter2_weights[0, 3], filter2_weights[1, 11], filter2_weights[2, 22], filter2_weights[3, 31], filter2_weights[4, 39], filter2_weights[5, 47], filter2_weights[6, 55], filter2_weights[7, 63] = 1, 1, 1, 1, 1, 1, 1, 1
        filter3_weights[0, 3], filter3_weights[1, 11], filter3_weights[2, 22], filter3_weights[3, 32], filter3_weights[4, 40], filter3_weights[5, 48], filter3_weights[6, 56], filter3_weights[7, 64] = 1, 1, 1, 1, 1, 1, 1, 1

        custom_weights = np.stack([filter1_weights, filter2_weights, filter3_weights], axis=0)
        self.conv2.weight.data = torch.tensor(custom_weights, dtype=torch.float32)
        self.conv2.weight.requires_grad = False
        self.conv2.bias.data.zero_()
        self.conv2.bias.requires_grad = True

    def energy(self, dna_seq):
        x = self.conv1(dna_seq)
        x = self.conv2(x)
        return torch.max(x.reshape(x.size(0), -1), dim=1, keepdim=True).values

    def energy2expression(self, x):
        beta = 1 / (0.001987 * 310)
        boltz_weight = torch.exp(beta * (x + self.param_e0))
        max_expr = torch.exp(self.param_max)
        min_expr = torch.exp(self.param_min)
        return torch.log(min_expr + max_expr * boltz_weight / (1 + boltz_weight))

    def forward(self, dna_seq, conds):
        x = self.energy(dna_seq)
        x = x + torch.matmul(conds, self.conds_bias.unsqueeze(1))
        return self.energy2expression(x)


## Train/test split and tensors


In [ ]:
TEST_SIZE = 0.2
STRATIFY_BY_LIBRARY = False
BATCH_SIZE = 32

library_categorical = pd.Categorical(df_train['Library'], categories=LIBRARY_ORDER)
conds = pd.get_dummies(library_categorical).to_numpy(dtype=np.float32)
dna_seq = np.array(list(df_train['Sequence'].apply(dna_one_hot)), dtype=np.float32)
target_loge = (df_train['LogGFP'].values * np.log(10)).astype(np.float32)
libraries = df_train['Library'].values
indices = np.arange(len(df_train))

stratify_labels = libraries if STRATIFY_BY_LIBRARY else None
split = train_test_split(
    dna_seq, conds, target_loge, libraries, indices,
    test_size=TEST_SIZE,
    random_state=42,
    stratify=stratify_labels,
)
(
    X_train_seq_np, X_test_seq_np,
    X_train_conds_np, X_test_conds_np,
    y_train_np, y_test_np,
    library_train, library_test,
    idx_train, idx_test,
) = split

X_train_seq = torch.tensor(X_train_seq_np, dtype=torch.float32, device=device)
X_train_conds = torch.tensor(X_train_conds_np, dtype=torch.float32, device=device)
y_train = torch.tensor(y_train_np, dtype=torch.float32, device=device).unsqueeze(1)

X_test_seq = torch.tensor(X_test_seq_np, dtype=torch.float32, device=device)
X_test_conds = torch.tensor(X_test_conds_np, dtype=torch.float32, device=device)
y_test = torch.tensor(y_test_np, dtype=torch.float32, device=device).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_seq, X_train_conds, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_seq, X_test_conds, y_test), batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(X_train_seq):,}; Test: {len(X_test_seq):,}; Sequence length: {dna_seq.shape[2]}")


## Training


In [ ]:
seq_length = dna_seq.shape[2]
num_conds = conds.shape[1]
model = DNAFunctionPredictor(seq_length=seq_length, num_conds=num_conds).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

EPOCHS = 50
USE_TENSORBOARD = False
TENSORBOARD_LOG_DIR = "runs/20250628_CorePromoter_clean"
writer = SummaryWriter(TENSORBOARD_LOG_DIR) if USE_TENSORBOARD else None
history = []

for epoch in range(EPOCHS):
    model.train()
    for seq_batch, cond_batch, target_batch in train_loader:
        optimizer.zero_grad()
        output = model(seq_batch, cond_batch)
        loss = criterion(output, target_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        train_loss = criterion(model(X_train_seq, X_train_conds) / np.log(10), y_train / np.log(10)).item()
        test_loss = criterion(model(X_test_seq, X_test_conds) / np.log(10), y_test / np.log(10)).item()

    history.append({'epoch': epoch + 1, 'train_mse_log10': train_loss, 'test_mse_log10': test_loss})
    if writer is not None:
        writer.add_scalars('Loss', {'Training Loss': train_loss, 'Testing Loss': test_loss}, epoch + 1)
    print(f"Epoch [{epoch + 1:02d}/{EPOCHS}] - Train MSE: {train_loss:.4f}, Test MSE: {test_loss:.4f}")

if writer is not None:
    writer.close()

history_df = pd.DataFrame(history)
display(history_df.tail())


## Save checkpoint

Overwrites `../weights/weights_CorePromoter_clean.pt` with this run. Several scripts and notebooks load that file as their default core model, so anything trained here becomes the shared baseline. Set `SAVE_CHECKPOINT = False` to explore without publishing the result.


In [ ]:
## Set to False to run the notebook without touching the shared checkpoint.
## Downstream consumers of the .pt:
##   automated_promoter_library_design.py             (load_core_model default)
##   train_corepromoter_tss_pas.py                    (DEFAULT_BASELINE)
##   library_release/01_recursive_design.ipynb        (variant "baseline")
##   Model_CorePromoter_energy_vs_conservation.ipynb
SAVE_CHECKPOINT = True

import json
from datetime import datetime

WEIGHTS_DIR = "../weights"
CHECKPOINT_PATH = f"{WEIGHTS_DIR}/weights_CorePromoter_clean.pt"

if SAVE_CHECKPOINT:
    os.makedirs(WEIGHTS_DIR, exist_ok=True)
    trained_at = datetime.now().isoformat(timespec='seconds')
    final_epoch = history_df.iloc[-1]

    torch.save({
        'model_state_dict': model.state_dict(),
        'seq_length': int(seq_length),
        'num_conds': int(num_conds),
        'seed': SEED,
        'trained_at': trained_at,
        'source': 'MS2_Data_PyTorch/scripts/Model_CorePromoter_clean.ipynb',
    }, CHECKPOINT_PATH)

    # Nothing reads these three, but they are the provenance record that ships
    # with the checkpoint; stale sidecars next to fresh weights are worse than none.
    history_df.to_csv(f"{WEIGHTS_DIR}/weights_CorePromoter_clean_history.csv", index=False)
    df_train.groupby('Library').size().rename('n').to_csv(
        f"{WEIGHTS_DIR}/weights_CorePromoter_clean_training_counts.csv"
    )
    with open(f"{WEIGHTS_DIR}/weights_CorePromoter_clean_metadata.json", 'w') as fh:
        json.dump({
            'seq_length': int(seq_length),
            'num_conds': int(num_conds),
            'seed': SEED,
            'trained_at': trained_at,
            'source': 'MS2_Data_PyTorch/scripts/Model_CorePromoter_clean.ipynb',
            'final_train_mse_log10': float(final_epoch['train_mse_log10']),
            'final_test_mse_log10': float(final_epoch['test_mse_log10']),
        }, fh, indent=2)

    # Read the .pt back the way the consumers do (weights_only=True) so a format
    # regression fails here rather than in a downstream design run.
    reloaded = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
    torch.testing.assert_close(
        reloaded['model_state_dict']['conv1.weight'],
        model.conv1.weight.detach().cpu(),
    )
    print(f"Saved {CHECKPOINT_PATH} (verified reload; trained_at {trained_at})")
    print(f"  final train MSE {final_epoch['train_mse_log10']:.4f}, "
          f"test MSE {final_epoch['test_mse_log10']:.4f}")
else:
    print(f"SAVE_CHECKPOINT is False; {CHECKPOINT_PATH} left unchanged")


In [ ]:
plt.plot(model.conv2.bias.cpu().detach().numpy())

In [ ]:
def ShowLogo(ax, pwm):
    ## custom color scheme
    color_scheme = {
        'A' : 'green',
        'C' : 'blue',
        'G' : 'orange',
        'T' : 'red',
        u'АСᏀТ': '#dfdfdf'
        }

    ## plot the Logo
    pwm_logo = logomaker.Logo(pwm,
                              shade_below=.8,
                              fade_below=.8,
                              font_name='DejaVu Sans',
                              color_scheme=color_scheme,
                              ax=ax)
    
    return None
df_motif1 = pd.DataFrame(np.array(model.conv1.weight.data.cpu()[0]).T, columns=['A', 'C', 'G', 'T'])
df_motif1 = df_motif1.sub(df_motif1.mean(axis=1), axis=0)  # Center each row
df_motif2 = pd.DataFrame(np.array(model.conv1.weight.data.cpu()[1]).T, columns=['A', 'C', 'G', 'T'])
df_motif2 = df_motif2.sub(df_motif2.mean(axis=1), axis=0)  # Center each row
df_motif3 = pd.DataFrame(np.array(model.conv1.weight.data.cpu()[2]).T, columns=['A', 'C', 'G', 'T'])
df_motif3 = df_motif3.sub(df_motif3.mean(axis=1), axis=0)  # Center each row
df_motif4 = pd.DataFrame(np.array(model.conv1.weight.data.cpu()[3]).T, columns=['A', 'C', 'G', 'T'])
df_motif4 = df_motif4.sub(df_motif4.mean(axis=1), axis=0)  # Center each row
df_motif5 = pd.DataFrame(np.array(model.conv1.weight.data.cpu()[4]).T, columns=['A', 'C', 'G', 'T'])
df_motif5 = df_motif5.sub(df_motif5.mean(axis=1), axis=0)  # Center each row
df_motif6 = pd.DataFrame(np.array(model.conv1.weight.data.cpu()[5]).T, columns=['A', 'C', 'G', 'T'])
df_motif6 = df_motif6.sub(df_motif6.mean(axis=1), axis=0)  # Center each row
df_motif7 = pd.DataFrame(np.array(model.conv1.weight.data.cpu()[6]).T, columns=['A', 'C', 'G', 'T'])
df_motif7 = df_motif7.sub(df_motif7.mean(axis=1), axis=0)  # Center each row
df_motif8 = pd.DataFrame(np.array(model.conv1.weight.data.cpu()[7]).T, columns=['A', 'C', 'G', 'T'])
df_motif8 = df_motif8.sub(df_motif8.mean(axis=1), axis=0)  # Center each row

fig, axes = plt.subplots(1, 8, figsize=(10, 1.5), dpi=150, sharey=True)

ShowLogo(axes[0], df_motif1)
ShowLogo(axes[1], df_motif2)
ShowLogo(axes[2], df_motif3)
ShowLogo(axes[3], df_motif4)
ShowLogo(axes[4], df_motif5)
ShowLogo(axes[5], df_motif6)
ShowLogo(axes[6], df_motif7)
ShowLogo(axes[7], df_motif8)





# ymin, ymax = ax.get_ylim()
# axes[0].vlines([5.5], ymin, ymax, color='black', lw=1)
# axes[1].vlines([7.5], ymin, ymax, color='black', lw=1)

axes[0].set_xticks([])
axes[1].set_xticks([])
axes[2].set_xticks([])
axes[3].set_xticks([])
axes[4].set_xticks([])
axes[5].set_xticks([])
axes[6].set_xticks([])
axes[7].set_xticks([])

axes[3].set_ylim(-3, 3)

plt.show()

In [ ]:
def build_spatial_logo_matrix(core_model, channel):
    """Arrange mean-centered conv1 weights at their true conv2 offsets."""
    bases = ['A', 'C', 'G', 'T']
    w1 = core_model.conv1.weight.detach().cpu().numpy()  # (filter, base, filter_pos)
    w2 = core_model.conv2.weight.detach().cpu().numpy()[channel]  # (filter, conv2_pos)

    if w1.shape[0] != w2.shape[0]:
        raise ValueError(f"conv1 has {w1.shape[0]} filters but conv2 channel {channel} has {w2.shape[0]}")
    if w1.shape[1] != len(bases):
        raise ValueError(f"expected {len(bases)} DNA bases, got conv1 shape {w1.shape}")

    offsets = []
    for filter_idx in range(w2.shape[0]):
        taps = np.flatnonzero(np.abs(w2[filter_idx]) > 1e-6)
        if taps.size != 1:
            raise ValueError(
                f"conv2 channel {channel}, filter {filter_idx} has {taps.size} nonzero taps; expected exactly 1"
            )
        offsets.append(int(taps[0]))

    centered = w1 - w1.mean(axis=1, keepdims=True)
    filter_width = w1.shape[2]
    first_pos = min(offsets)
    last_pos = max(offset + filter_width - 1 for offset in offsets)
    logo_matrix = pd.DataFrame(
        0.0,
        index=np.arange(last_pos - first_pos + 1),
        columns=bases,
    )
    occupied = np.zeros(len(logo_matrix), dtype=bool)

    for filter_idx, offset in enumerate(offsets):
        start = offset - first_pos
        stop = start + filter_width
        if occupied[start:stop].any():
            overlap = np.flatnonzero(occupied[start:stop]) + start
            raise ValueError(
                f"conv2 channel {channel}, filter {filter_idx} overlaps logo positions {overlap.tolist()}"
            )
        logo_matrix.iloc[start:stop, :] = centered[filter_idx].T
        occupied[start:stop] = True

        np.testing.assert_allclose(
            logo_matrix.iloc[start:stop, :].to_numpy(),
            centered[filter_idx].T,
        )

    gap_positions = np.flatnonzero(~occupied)
    np.testing.assert_allclose(logo_matrix.iloc[gap_positions, :].to_numpy(), 0.0)
    return logo_matrix, offsets, gap_positions.tolist()


spacer_channels = {16: 0, 17: 1, 18: 2}
expected_logo_lengths = {16: 67, 17: 68, 18: 69}

for spacer, channel in spacer_channels.items():
    spatial_logo, offsets, gap_positions = build_spatial_logo_matrix(model, channel)
    assert len(spatial_logo) == expected_logo_lengths[spacer], (
        f"spacer {spacer}: expected {expected_logo_lengths[spacer]} positions, got {len(spatial_logo)}"
    )

    fig, ax = plt.subplots(figsize=(18, 2.4), dpi=150)
    ShowLogo(ax, spatial_logo)
    ax.set_xticks([])
    ax.set_ylim(-3, 3)
    ax.set_title(
        f"Whole model weights — spacer {spacer} "
        f"( blank columns = uncovered gaps)"
    )
    fig.tight_layout()
    plt.show()

    print(
        f"spacer {spacer}: {len(spatial_logo)} bp, "
        f"offsets={offsets}, gaps={gap_positions}"
    )

In [ ]:
def build_element_segmented_logo(core_model, spacer, channel):
    """Map the conv2-arranged weights onto UP/-35/spacer/-10/Dis/Start/ITR coordinates."""
    panel_specs = [
        ('UP', 20),
        ('-35', 6),
        ('Spacer', spacer),
        ('-10', 6),
        ('Dis', 5),
        ('Start', 3),
        ('ITR', 20),
    ]

    spatial_logo, offsets, spatial_gap_positions = build_spatial_logo_matrix(core_model, channel)
    m35_filter_idx = 2
    m35_length = 6
    m10_start = offsets[m35_filter_idx] + m35_length + spacer

    rel_start = -(20 + m35_length + spacer)
    total_width = sum(width for _, width in panel_specs)
    rel_positions = np.arange(rel_start, rel_start + total_width)
    full_logo = pd.DataFrame(0.0, index=rel_positions, columns=['A', 'C', 'G', 'T'])

    first_conv2_pos = min(offsets)
    for local_pos, values in spatial_logo.iterrows():
        rel_pos = first_conv2_pos + int(local_pos) - m10_start
        if rel_pos not in full_logo.index:
            raise ValueError(
                f"spacer {spacer}, channel {channel}: model position {rel_pos} falls outside the segmented frame"
            )
        full_logo.loc[rel_pos] = values.to_numpy()

    # Validate that mapping into biological coordinates preserves every model-weight row.
    mapped = full_logo.loc[
        first_conv2_pos - m10_start:
        first_conv2_pos - m10_start + len(spatial_logo) - 1
    ].to_numpy()
    np.testing.assert_allclose(mapped, spatial_logo.to_numpy())

    panels = []
    cursor = 0
    for name, width in panel_specs:
        panel = full_logo.iloc[cursor:cursor + width].reset_index(drop=True)
        assert len(panel) == width
        panels.append((name, width, panel))
        cursor += width
    assert cursor == len(full_logo)

    spatial_gap_positions = set(spatial_gap_positions)
    covered_rel_positions = {
        first_conv2_pos + local_pos - m10_start
        for local_pos in range(len(spatial_logo))
        if local_pos not in spatial_gap_positions
    }
    uncovered_rel_positions = [
        int(rel_pos) for rel_pos in full_logo.index
        if rel_pos not in covered_rel_positions
    ]
    return panels, full_logo, uncovered_rel_positions


expected_segmented_lengths = {16: 76, 17: 77, 18: 78}

for spacer, channel in spacer_channels.items():
    panels, segmented_logo, uncovered_rel_positions = build_element_segmented_logo(
        model, spacer, channel
    )
    assert len(segmented_logo) == expected_segmented_lengths[spacer], (
        f"spacer {spacer}: expected {expected_segmented_lengths[spacer]} positions, "
        f"got {len(segmented_logo)}"
    )

    widths = [width for _, width, _ in panels]
    fig, axes = plt.subplots(
        1,
        len(panels),
        figsize=(18, 2.8),
        dpi=150,
        sharey=False,
        gridspec_kw={'width_ratios': widths, 'wspace': 0.16},
    )

    for ax, (name, width, panel) in zip(axes, panels):
        ShowLogo(ax, panel)
        ax.set_title(f"{name} ({width} bp)", fontsize=9)
        if width >= 10:
            middle = (width - 1) // 2
            ticks = [0, middle, width - 1]
        else:
            ticks = [0, width - 1]
        ax.set_xticks(ticks)
        ax.set_xticklabels([tick + 1 for tick in ticks], fontsize=7)
        ax.tick_params(axis='y', labelsize=7)
        ax.grid(axis='x', linestyle='--', linewidth=0.5, alpha=0.45)

    fig.suptitle(
        f"CorePromoter mean-centered conv1 weights by element — "
        f"spacer {spacer} (conv2 channel {channel})",
        fontsize=11,
    )
    fig.subplots_adjust(top=0.78, bottom=0.18)
    plt.show()

    print(
        f"spacer {spacer}: panel widths={widths}, "
        f"uncovered rel_pos={uncovered_rel_positions}"
    )